## 1. Đọc dữ liệu

In [51]:
import pandas as pd
from pathlib import Path

# Sửa PROJECT_DIR nếu notebook không nằm trong thư mục project.
PROJECT_DIR = Path.cwd().resolve()

# Nếu notebook được chạy ở thư mục con, tìm thư mục project có 01_craw_data.
if not (PROJECT_DIR / "01_craw_data").is_dir():
    for parent in PROJECT_DIR.parents:
        if (parent / "01_craw_data").is_dir():
            PROJECT_DIR = parent
            break

DATA_DIR = PROJECT_DIR / "01_craw_data" 

category = pd.read_csv(DATA_DIR / 'category' / 'category.csv', encoding='utf-8')
product = pd.read_csv(DATA_DIR / 'product' / 'product.csv', encoding='utf-8')
product_detail = pd.read_csv(DATA_DIR / 'product' / 'product_detail.csv', encoding='utf-8')

product_pareto = DATA_DIR / "daily_price" / "product_pareto.csv"

print('Danh mục:', category.shape[0])
print('Số lượng sản phẩm:', product_detail.shape[0])

display(category.head())
display(product.head())
display(product_detail.head())

Danh mục: 26
Số lượng sản phẩm: 42365


,category_id,name,urlKey,src
0,8322,Nhà Sách Tiki,nha-sach-tiki,c8322
1,1883,Nhà Cửa - Đời Sống,nha-cua-doi-song,c1883
2,1789,Điện Thoại - Máy Tính Bảng,dien-thoai-may-tinh-bang,c1789
3,2549,Đồ Chơi - Mẹ & Bé,do-choi-me-be,c2549
4,1815,Thiết Bị Số - Phụ Kiện Số,thiet-bi-kts-phu-kien-so,c1815


,product_id,seller_product_id,category_id,crawl_date
0,278496130,278496163,8322,2026-10
1,278777861,278826448,8322,2026-10
2,278790552,278790687,8322,2026-10
3,278856619,278856627,8322,2026-10
4,278785576,92281,8322,2026-10


,product_id,master_id,sku,product_name,product_type,url_key,created_date,favourite_count,brand_id,brand_name,rating_average,review_count,all_time_quantity_sold,crawl_date
0,278496130,278496130,6286902619121,Thanh Gươm Diệt Quỷ,NaN,thanh-guom-diet-quy-tap-1-tan-khoc-p278496130,2025-08-13,0,NaN,NaN,5.0,531,251.0,2026-10
1,278777861,278777861,6635841683218,Bộ Sách Quiz! Khoa Học Kì Thú [Chọn Tập Lẻ],NaN,bo-sach-quiz-khoa-hoc-ki-thu-p278777861,2025-10-20,0,NaN,NaN,5.0,38,1200.0,2026-10
2,278790552,278790552,5178819868028,Bộ Truyện One Piece [Chọn Tập Lẻ],NaN,bo-truyen-one-piece-chon-tap-le-p278790552,2025-10-22,0,NaN,NaN,5.0,288,685.0,2026-10
3,278856619,278856619,8181521199193,Sách Cô Bé Hàng Xóm Và Bốn Viên Kẹo (Nguyễn Nh...,NaN,sach-co-be-hang-xom-va-bon-vien-keo-p278856619,2025-11-10,0,NaN,NaN,5.0,75,928.0,2026-10
4,278785576,278785576,6844760652808,Sách Dragon Ball - 7 Viên Ngọc Rồng [Chọn Tập Lẻ],NaN,sach-dragon-ball-7-vien-ngoc-rong-chon-tap-le-...,2025-10-20,0,NaN,NaN,4.8,4,409.0,2026-10


# 2. Chuẩn hóa dữ liệu và Merge product với product_detail

In [52]:
product_detail = product_detail[
    ["product_id", "all_time_quantity_sold"]
]

product_detail.fillna(0, inplace=True)

df = product_detail.merge(
    product[["product_id", "category_id"]],
    on="product_id",
    how="left"
).merge(
    category[["category_id", "name"]],
    on="category_id",
    how="left"
).drop_duplicates()

display(df.head())
display(df.info())

,product_id,all_time_quantity_sold,category_id,name
0,278496130,251.0,8322,Nhà Sách Tiki
1,278777861,1200.0,8322,Nhà Sách Tiki
2,278790552,685.0,8322,Nhà Sách Tiki
3,278856619,928.0,8322,Nhà Sách Tiki
4,278785576,409.0,8322,Nhà Sách Tiki


<class 'pandas.DataFrame'>
Index: 42365 entries, 0 to 42367
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   product_id              42365 non-null  int64  
 1   all_time_quantity_sold  42365 non-null  float64
 2   category_id             42365 non-null  int64  
 3   name                    42365 non-null  str    
dtypes: float64(1), int64(2), str(1)
memory usage: 2.5 MB


None

# 3. Phân tích pareto danh mục theo all_time_quantity_sold

In [53]:
df_pareto = (
    df.groupby(["category_id", "name"], as_index=False)
    .agg(total_quantity_sold=("all_time_quantity_sold", "sum"))
    .sort_values("total_quantity_sold", ascending=False)
    .reset_index()
)

# Tính tỷ trọng doanh số
total_sold = df_pareto["total_quantity_sold"].sum()

df_pareto["sales_share"] = (
    df_pareto["total_quantity_sold"] / total_sold * 100
)

# Tính tỷ trọng lũy kế
df_pareto["cumulative_share"] = df_pareto["sales_share"].cumsum()

# Phân nhóm Pareto: nhóm danh mục đóng góp đến 80%
df_pareto["pareto_group"] = (
    df_pareto["cumulative_share"] <= 80
).map({True: "Top 80%", False: "Remaining 20%"})

display(df_pareto)

,index,category_id,name,total_quantity_sold,sales_share,cumulative_share,pareto_group
0,3,1520,Làm Đẹp - Sức Khỏe,5385715.0,31.053912,31.053912,Top 80%
1,17,8322,Nhà Sách Tiki,3368271.0,19.421375,50.475287,Top 80%
2,20,11312,Voucher - Dịch vụ,2725499.0,15.715166,66.190453,Top 80%
3,13,2549,Đồ Chơi - Mẹ & Bé,2012064.0,11.601516,77.791968,Top 80%
4,15,4384,Bách Hóa Online,1864673.0,10.751663,88.543631,Remaining 20%
5,11,1883,Nhà Cửa - Đời Sống,512538.0,2.955283,91.498914,Remaining 20%
6,10,1882,Điện Gia Dụng,311337.0,1.795162,93.294076,Remaining 20%
7,8,1815,Thiết Bị Số - Phụ Kiện Số,272605.0,1.571834,94.865910,Remaining 20%
8,21,15078,Chăm sóc nhà cửa,220496.0,1.271375,96.137285,Remaining 20%
9,9,1846,Laptop - Máy Vi Tính - Linh kiện,147378.0,0.849778,96.987063,Remaining 20%


# 4. Lấy các product_id thuộc danh mục top 80% để theo dõi giá hằng ngày

In [54]:
# Lấy danh sách category_id thuộc nhóm Top 80%
top_categories = df_pareto.loc[
    df_pareto["pareto_group"] == "Top 80%",
    "category_id"
]

# Lấy danh sách product_id thuộc các danh mục Top 80%
df_product_top80 = (
    df.loc[df["category_id"].isin(top_categories), ["product_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Merge với product để lấy đầy đủ thông tin
df_product_top80 = df_product_top80.merge(
    product[["product_id", "seller_product_id", "category_id"]],
    on="product_id",
    how="left"
)
display(df_product_top80.head())

,product_id,seller_product_id,category_id
0,278496130,278496163,8322
1,278777861,278826448,8322
2,278790552,278790687,8322
3,278856619,278856627,8322
4,278785576,92281,8322


# 5. Lưu vào thư mục 01_craw_data/daily_price để craw giá hằng ngày

In [55]:
# Nếu thư mục 01_craw_data/daily đã có file product_pareto.csv thì không lưu
# Chủ yếu theo dõi các sản phẩm thuộc top 80% danh mục T10
# Chỉ lưu nếu file chưa tồn tại
if not product_pareto.exists():
    df_product_top80.to_csv(
        product_pareto,
        index=False,
        encoding="utf-8-sig"
    )
    print(f"Đã lưu: {product_pareto}")
else:
    print(f"File đã tồn tại, bỏ qua lưu: {product_pareto}")

File đã tồn tại, bỏ qua lưu: D:\TiKi-Analysic\01_craw_data\daily_price\product_pareto.csv
